In [6]:
import os
import time
import requests
import pandas as pd
from google.colab import drive


print("Mounting Google Drive...")
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Datastorm'
bronze_path = os.path.join(base_path, 'Bronze')
os.makedirs(bronze_path, exist_ok=True)
print(f"Lakehouse Architecture Confirmed at: {base_path}\n")

def scrape_osm_poi(tag_key, tag_value, poi_name, bbox="(5.9, 79.5, 9.9, 81.9)"):
    print(f"Fetching {poi_name} data from OpenStreetMap...")
    url = "http://overpass-api.de/api/interpreter"

    query = f"""
    [out:json][timeout:90];
    (
      node["{tag_key}"="{tag_value}"]{bbox};
    );
    out center;
    """

    headers = {'User-Agent': 'DataStorm7_Hackathon_Project_SriLanka'}

    try:
        response = requests.post(url, headers=headers, data={'data': query})
        response.raise_for_status()

        data = response.json()
        poi_list = []
        for element in data.get('elements', []):
            poi_list.append({
                'POI_ID': element.get('id'),
                'POI_Type': poi_name,
                'Name': element.get('tags', {}).get('name', 'Unknown'),
                'Latitude': element.get('lat'),
                'Longitude': element.get('lon')
            })

        df = pd.DataFrame(poi_list)
        print(f" -> Success: Extracted {len(df)} records for {poi_name}.")
        return df

    except requests.exceptions.RequestException as e:
        print(f" -> Error fetching {poi_name}: {e}")
        return pd.DataFrame()

poi_targets = [
    {'key': 'amenity', 'value': 'school', 'name': 'School'},
    {'key': 'amenity', 'value': 'hospital', 'name': 'Hospital'},
    {'key': 'highway', 'value': 'bus_stop', 'name': 'Bus_Stop'},
    {'key': 'amenity', 'value': 'bus_station', 'name': 'Bus_Station'},
    {'key': 'amenity', 'value': 'marketplace', 'name': 'Marketplace'}
]

scraped_dfs = []


for target in poi_targets:
    df = scrape_osm_poi(target['key'], target['value'], target['name'])
    if not df.empty:
        scraped_dfs.append(df)
    time.sleep(2)

if scraped_dfs:
    master_poi_df = pd.concat(scraped_dfs, ignore_index=True)
    print(f"\nTotal POIs Scraped: {len(master_poi_df)}")

    poi_export_path = os.path.join(bronze_path, 'external_scraped_pois.csv')
    master_poi_df.to_csv(poi_export_path, index=False)
    print(f"Saved raw external POI data to Bronze Layer: {poi_export_path}")

    display(master_poi_df.head())
else:
    print("\nNo POI data was extracted. Check your internet connection or API status.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Lakehouse Architecture Confirmed at: /content/drive/MyDrive/Datastorm

Fetching School data from OpenStreetMap...
 -> Success: Extracted 2390 records for School.
Fetching Hospital data from OpenStreetMap...
 -> Success: Extracted 459 records for Hospital.
Fetching Bus_Stop data from OpenStreetMap...
 -> Success: Extracted 3432 records for Bus_Stop.
Fetching Bus_Station data from OpenStreetMap...
 -> Success: Extracted 258 records for Bus_Station.
Fetching Marketplace data from OpenStreetMap...
 -> Success: Extracted 145 records for Marketplace.

Total POIs Scraped: 6684
Saved raw external POI data to Bronze Layer: /content/drive/MyDrive/Datastorm/Bronze/external_scraped_pois.csv


,POI_ID,POI_Type,Name,Latitude,Longitude
0,64638677,School,Sanghamitta College,6.047125,80.212170
1,64638681,School,Mahinda College,6.049469,80.215119
2,77844735,School,Ceylinco Sussex College - Bandarawella,6.839086,80.977190
3,91029264,School,Ceylinco Sussex College - Galle,6.056283,80.204397
4,138741738,School,Sacred Heart Convent,6.036599,80.211026


In [7]:
poi_type_counts = master_poi_df['POI_Type'].value_counts()
print("Counts of each POI Type:")
print(poi_type_counts)

Counts of each POI Type:
POI_Type
Bus_Stop       3432
School         2390
Hospital        459
Bus_Station     258
Marketplace     145
Name: count, dtype: int64
